<a href="https://colab.research.google.com/github/Adeel213/Twitter-Sentiment-Analysis/blob/main/Sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import re
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Bidirectional

In [ ]:
# 1. Load data — local downloaded CSV, no header row
COLUMN_NAMES = ['tweet_id', 'entity', 'sentiment', 'text']
df = pd.read_csv('/content/drive/MyDrive/Data/twitter_training.csv', header=None, names=COLUMN_NAMES)


In [ ]:
# 2. Clean
def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = df.dropna(subset=['text']).reset_index(drop=True)
df['clean_text'] = df['text'].apply(clean_tweet)
df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)

In [ ]:
# 3. Encode labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['sentiment'])

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

In [ ]:
# 4. Pipeline — TF-IDF + LinearSVC bundled as ONE object
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
    ('clf', LinearSVC(max_iter=2000))
])

svm_pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
                ('clf', LinearSVC(max_iter=2000))])

In [ ]:
# 5. Evaluate
y_pred = svm_pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

Accuracy: 0.8436
              precision    recall  f1-score   support

  Irrelevant       0.85      0.77      0.81      2562
    Negative       0.86      0.88      0.87      4451
     Neutral       0.84      0.83      0.84      3596
    Positive       0.82      0.86      0.84      4116

    accuracy                           0.84     14725
   macro avg       0.84      0.84      0.84     14725
weighted avg       0.84      0.84      0.84     14725



In [ ]:
# 6. Save — one pipeline file + label encoder, ready for the frontend
with open('sentiment_svm_pipeline.pkl', 'wb') as f:
    pickle.dump(svm_pipeline, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

print("Saved: sentiment_svm_pipeline.pkl, label_encoder.pkl")

Saved: sentiment_svm_pipeline.pkl, label_encoder.pkl


Simple RNN


In [ ]:
NUM_CLASSES = len(label_encoder.classes_)

# 1. Data-driven MAX_LEN — same 95th-percentile approach
word_counts = X_train.str.split().apply(len)
VOCAB_SIZE = 20000
MAX_LEN = int(word_counts.quantile(0.95))
print(f"MAX_LEN (95th percentile): {MAX_LEN}")

MAX_LEN (95th percentile): 48


In [ ]:
# 2. TextVectorization adapted on training text only
vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN
)
vectorize_layer.adapt(X_train.values)

In [ ]:
# 3. tf.data pipelines
BATCH_SIZE = 64

train_ds = tf.data.Dataset.from_tensor_slices(
    (X_train.values, y_train.values)
).shuffle(10000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(
    (X_test.values, y_test.values)
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [ ]:
# 4. Model — TextVectorization lives INSIDE the model
EMBEDDING_DIM = 128
RNN_UNITS = 64

rnn_model = tf.keras.Sequential([
    tf.keras.Input(shape=(), dtype=tf.string),
    vectorize_layer,
    tf.keras.layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, mask_zero=True),
    Bidirectional(SimpleRNN(128, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(SimpleRNN(64)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(4, activation='softmax')
])


In [ ]:
rnn_model.compile(optimizer='adam',
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])
rnn_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 48)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_3 (Embedding)         │ (None, 48, 128)        │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 48, 256)        │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 48, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,675,396 (10.21 MB)

 Trainable params: 2,675,396 (10.21 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 5. Train
early_stopping = EarlyStopping(monitor='val_accuracy', mode='max', patience=5, restore_best_weights=True)

history = rnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[early_stopping],
    verbose=2
)


Epoch 1/30
921/921 - 171s - 186ms/step - accuracy: 0.5982 - loss: 0.9646 - val_accuracy: 0.7835 - val_loss: 0.5916
Epoch 2/30
921/921 - 235s - 255ms/step - accuracy: 0.8544 - loss: 0.3986 - val_accuracy: 0.8539 - val_loss: 0.4236
Epoch 3/30
921/921 - 204s - 222ms/step - accuracy: 0.9149 - loss: 0.2341 - val_accuracy: 0.8697 - val_loss: 0.3819
Epoch 4/30
921/921 - 195s - 212ms/step - accuracy: 0.9351 - loss: 0.1784 - val_accuracy: 0.8795 - val_loss: 0.3795
Epoch 5/30
921/921 - 195s - 212ms/step - accuracy: 0.9438 - loss: 0.1497 - val_accuracy: 0.8776 - val_loss: 0.4044
Epoch 6/30
921/921 - 196s - 213ms/step - accuracy: 0.9500 - loss: 0.1302 - val_accuracy: 0.8846 - val_loss: 0.3861
Epoch 7/30
921/921 - 200s - 217ms/step - accuracy: 0.9528 - loss: 0.1225 - val_accuracy: 0.8803 - val_loss: 0.4123
Epoch 8/30
921/921 - 202s - 220ms/step - accuracy: 0.9562 - loss: 0.1141 - val_accuracy: 0.8820 - val_loss: 0.4133
Epoch 9/30
921/921 - 212s - 230ms/step - accuracy: 0.9579 - loss: 0.1075 - val_a

In [ ]:
# 6. Best epoch result
best_epoch = int(np.argmax(history.history['val_accuracy']))
print(f"Best epoch: {best_epoch + 1}")
print(f"SimpleRNN validation accuracy: {history.history['val_accuracy'][best_epoch]:.4f}")

Best epoch: 17
SimpleRNN validation accuracy: 0.8947


In [ ]:
results = {}
# 7. Add to results dict (if you're tracking ML model scores in `results`)
results['SimpleRNN'] = history.history['val_accuracy'][best_epoch]

In [ ]:
# 8. Save — .keras, not .pkl
rnn_model.save('sentiment_rnn_model.keras')
print("Saved: sentiment_rnn_model.keras")

Saved: sentiment_rnn_model.keras


In [ ]:
from google.colab import files
files.download('sentiment_rnn_model1.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
rnn_model.save("sentiment_rnn_model1.keras", include_optimizer=False)